# Hate Speech Detection — Full Pipeline Demo

**Architecture (no routing):**
```
text → [Layer 2] retrieve neighbors → augment → RAG classifier → label + confidence
     → [Layer 3] LLM explanation → structured moderation output
```

Evaluated on 50 real 4chan posts with ground-truth labels (`hatespeech_dataset_4chan.xlsx`).  
Edit **Cell 3** to switch model configuration.

In [1]:
import sys, os, json, re, torch, faiss
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

sys.path.insert(0, str(Path(".").resolve()))  # makes rag.py and layer3_explainer.py importable
from rag import retrieve_top_k, retrieve_top_k_above_threshold
from layer3_explainer import explain, Layer2Output

ModuleNotFoundError: No module named 'faiss'

## Cell 2 — LLM Backend

In [ ]:
# --- Option A: Groq (free, recommended) — get a key at console.groq.com ---
from groq import Groq
llm_client = Groq(api_key=os.getenv("GROQ_API_KEY", ""))
LLM_MODEL = "llama3-8b-8192"

# --- Option B: Ollama (local, free) ---
# import openai
# llm_client = openai.OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# LLM_MODEL = "mistral"

# --- Option C: OpenAI (paid) ---
# import openai
# llm_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY", ""))
# LLM_MODEL = "gpt-4o-mini"

print(f"LLM backend ready: {LLM_MODEL}")

## Cell 3 — Pipeline Config
`INDEX_SPLIT` controls both which FAISS index is queried and which classifier weights are loaded — they must match.

In [ ]:
MODEL_FAMILY = "roberta"   # "bert" | "hatebert" | "roberta"
INDEX_SPLIT  = "training"  # "training" | "documents" | "full"
DATASET      = "ISHate"    # "IHC" | "ISHate" | "Vicomtech"
K            = 5           # max neighbors to retrieve
THRESHOLD    = 0.98        # min cosine similarity; set to 0.0 to always get K neighbors

## Cell 4 — Load Dataset

In [ ]:
df = pd.read_excel("../hatespeech_dataset_4chan.xlsx")
print(f"Dataset: {len(df)} examples")
print(f"Label distribution: {df['label'].value_counts().to_dict()}")
df.head()

## Cell 5 — Load Pipeline Components

In [ ]:
HF_IDS = {
    "bert":     "bert-base-uncased",
    "hatebert": "GroNLP/hateBERT",
    "roberta":  "roberta-base",
}

def load_pipeline(model_family, index_split, dataset):
    hf_id  = HF_IDS[model_family]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    print(f"Loading retriever: {hf_id} ...")
    ret_tokenizer = AutoTokenizer.from_pretrained(hf_id)
    ret_model     = AutoModel.from_pretrained(hf_id).eval().to(device)

    index_path  = f"index/{model_family}/base/vdb_{index_split}.faiss"
    lookup_path = f"index/lookup_{index_split}.json"
    print(f"Loading index: {index_path} ...")
    index = faiss.read_index(index_path)
    with open(lookup_path) as f:
        documents = json.load(f)
    print(f"  Index size: {index.ntotal:,} vectors")

    clf_path = f"../weights_rag/{model_family}/base/{index_split}/{dataset}"
    print(f"Loading RAG classifier: {clf_path} ...")
    clf_tokenizer = AutoTokenizer.from_pretrained(clf_path)
    clf_model     = AutoModelForSequenceClassification.from_pretrained(clf_path).eval().to(device)

    print(f"\nReady: {model_family.upper()} | index={index_split} | trained_on={dataset}")
    return ret_model, ret_tokenizer, index, documents, clf_model, clf_tokenizer, device


ret_model, ret_tokenizer, index, documents, clf_model, clf_tokenizer, device = load_pipeline(
    MODEL_FAMILY, INDEX_SPLIT, DATASET
)

## Cell 6 — Pipeline Runner and Display

In [ ]:
_LABEL_RE = re.compile(r"^\[(hate|not hate)\]\s*:?\s*", re.IGNORECASE)

def strip_label(text):
    return _LABEL_RE.sub("", text).strip()


def run_pipeline(text, ret_model, ret_tokenizer, index, documents,
                 clf_model, clf_tokenizer, device,
                 llm_client, llm_model, k=K, threshold=THRESHOLD):

    # Layer 2a: retrieve neighbors
    retrieved = retrieve_top_k_above_threshold(
        text, threshold, ret_model, ret_tokenizer, index, documents, chunk_id=None, k=k
    )
    if not retrieved:  # fallback: nothing cleared the threshold
        retrieved = retrieve_top_k(
            text, ret_model, ret_tokenizer, index, documents, chunk_id=None, k=k
        )

    # Layer 2b: augment and classify
    sep = clf_tokenizer.sep_token or "[SEP]"
    augmented = f" {sep} ".join([text] + [t for t, _ in retrieved])
    inputs = clf_tokenizer(
        augmented, return_tensors="pt", truncation=True, padding=True, max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = clf_model(**inputs).logits[0]
    probs      = F.softmax(logits, dim=-1)
    pred_idx   = torch.argmax(probs).item()
    label      = "hate" if pred_idx == 1 else "not hate"
    confidence = probs[pred_idx].item()

    # Layer 3: LLM explanation
    l2 = Layer2Output(
        original_text=text, label=label, confidence=confidence,
        hate_category="unknown", retrieved=retrieved
    )
    explanation = explain(l2, llm_client, llm_model)
    return l2, explanation


def display_result(example_id, text, ground_truth, l2, explanation):
    w = 80
    correct  = l2.label == ground_truth
    mark     = "\u2713" if correct else "\u2717"
    print("=" * w)
    print(f"[{example_id}] {mark}  TEXT : {text}")
    print(f"       GROUND TRUTH : {ground_truth.upper()}")
    print("-" * w)
    print(f"LAYER 2  : {l2.label.upper()}  ({l2.confidence:.1%} confidence)")
    print()
    print(f"RETRIEVED NEIGHBORS ({len(l2.retrieved)}):")
    for i, (txt, score) in enumerate(l2.retrieved, 1):
        print(f"  [{i}] {score:.4f}  {strip_label(txt)[:100]}")
    print()
    print("LAYER 3 EXPLANATION:")
    print(f"  Summary   : {explanation.summary}")
    print(f"  Severity  : {explanation.severity}")
    print(f"  Action    : {explanation.recommended_action}")
    print(f"  Targets   : {', '.join(explanation.target_groups) if explanation.target_groups else chr(8212)}")
    print(f"  Evidence  : {explanation.evidence_used}")
    if explanation.moderator_note:
        print(f"  Note      : {explanation.moderator_note}")
    valid_str = "\u2713 passed" if explanation.validation_passed else "\u2717 FAILED (forced human-review)"
    print(f"  Validation: {valid_str}")
    print("=" * w)
    print()

## Cell 7 — Run Full Pipeline on All 50 Examples

In [ ]:
print(f"Config : {MODEL_FAMILY.upper()} | index={INDEX_SPLIT} | trained_on={DATASET}")
print(f"Dataset: 4chan ({len(df)} examples)\n")

records = []

for _, row in df.iterrows():
    text         = str(row["text"])
    ground_truth = str(row["label"]).strip().lower()
    example_id   = int(row["id"])

    l2, exp = run_pipeline(
        text,
        ret_model, ret_tokenizer, index, documents,
        clf_model, clf_tokenizer, device,
        llm_client, LLM_MODEL,
    )
    display_result(example_id, text, ground_truth, l2, exp)

    records.append({
        "id"               : example_id,
        "text"             : text,
        "ground_truth"     : ground_truth,
        "predicted"        : l2.label,
        "confidence"       : round(l2.confidence, 4),
        "n_retrieved"      : len(l2.retrieved),
        "top_sim"          : round(l2.retrieved[0][1], 4) if l2.retrieved else None,
        "evidence_used"    : exp.evidence_used,
        "severity"         : exp.severity,
        "action"           : exp.recommended_action,
        "target_groups"    : exp.target_groups,
        "validation_passed": exp.validation_passed,
        "correct"          : l2.label == ground_truth,
    })

results_df = pd.DataFrame(records)
print(f"\nDone. {results_df['correct'].sum()}/{len(results_df)} correct.")

## Cell 8 — Classification Metrics

In [ ]:
y_true = results_df["ground_truth"].tolist()
y_pred = results_df["predicted"].tolist()
labels = ["hate", "not hate"]

print(f"Config : {MODEL_FAMILY.upper()} | index={INDEX_SPLIT} | trained_on={DATASET}")
print(f"Dataset: 4chan ({len(y_true)} examples)")
print("=" * 50)
print(f"Accuracy  : {accuracy_score(y_true, y_pred):.3f}")
print(f"F1 macro  : {f1_score(y_true, y_pred, average='macro'):.3f}")
print(f"Precision : {precision_score(y_true, y_pred, average='macro'):.3f}")
print(f"Recall    : {recall_score(y_true, y_pred, average='macro'):.3f}")
print()
print(classification_report(y_true, y_pred, target_names=labels))

cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(
    cm,
    index=[f"True: {l}" for l in labels],
    columns=[f"Pred: {l}" for l in labels]
)
print("Confusion matrix:")
display(cm_df)

## Cell 9 — Results Summary Table

In [ ]:
display(results_df[[
    "id", "ground_truth", "predicted", "confidence",
    "n_retrieved", "top_sim", "severity", "action",
    "validation_passed", "correct"
]])

## Cell 10 — Error Analysis

In [ ]:
errors = results_df[~results_df["correct"]].reset_index(drop=True)
print(f"Misclassified: {len(errors)}/{len(results_df)}\n")
for _, row in errors.iterrows():
    print(f"[{int(row['id'])}] GT={row['ground_truth'].upper():8s}  PRED={row['predicted'].upper():8s}  conf={row['confidence']:.1%}")
    print(f"       {row['text'][:120]}")
    print()

## Cell 11 — Compare All 27 Configs *(optional — slow, ~45 min on CPU)*

Runs every `(model, index_split, dataset)` combination on the full 4chan dataset and ranks by F1.

In [ ]:
ALL_CONFIGS = [
    (model, split, dataset)
    for model   in ["bert", "hatebert", "roberta"]
    for split   in ["training", "documents", "full"]
    for dataset in ["IHC", "ISHate", "Vicomtech"]
]

summary_rows = []

for model_f, split, dset in ALL_CONFIGS:
    config_name = f"{model_f}/{split}/{dset}"
    print(f"Running {config_name} ...", end=" ", flush=True)
    try:
        r_model, r_tok, idx, docs, c_model, c_tok, dev = load_pipeline(model_f, split, dset)
        preds, truths = [], []
        for _, row in df.iterrows():
            l2, _ = run_pipeline(
                str(row["text"]), r_model, r_tok, idx, docs, c_model, c_tok, dev,
                llm_client, LLM_MODEL
            )
            preds.append(l2.label)
            truths.append(str(row["label"]).strip().lower())

        acc = accuracy_score(truths, preds)
        f1  = f1_score(truths, preds, average="macro")
        summary_rows.append({"Config": config_name, "Accuracy": round(acc, 3), "F1 macro": round(f1, 3)})
        print(f"acc={acc:.3f}  f1={f1:.3f}")

        del r_model, c_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        print(f"ERROR: {e}")

summary_df = pd.DataFrame(summary_rows).sort_values("F1 macro", ascending=False)
display(summary_df)